# Part 3: Topics

In Part 2, the entity graph connected named things, such as a batch and the farms behind it, through typed relationships. Each relationship stated an explicit fact that you could trace back to source text. **Topics** answer a different question: what does this content cover, and where should I look?

A topic is a group of chunks with similar meaning. The topics were built from chunk embeddings when the lab data was prepared. Chunks with similar embeddings were clustered into the same group. Being in a topic means that chunks are similar. It doesn't mean that they state a verified fact about one another.

<img src="../../img/diagrams/notebook-03-topics-overview.svg" alt="Diagram in three steps. 1: Chunks of the original PDFs are compared by their embeddings, and similar chunks are connected; they are not entities or facts. 2: Similar chunks form a broad topic at level 0 with narrow topics A and B at level 1, plus a broad topic with no children. Each terminal topic has a centroid, or representative vector, and the broad topic's centroid is derived from its children. Similarity groups chunks; it does not assert facts. 3: The Sitemap leads from a broad topic to a narrower topic. A centroid search over all source chunks returns centroid-nearest examples for browsing, not stored membership. The original chunks and source records are the evidence. Topics guide where to look; source chunks provide the evidence." width="1000">

The topics form a two-level hierarchy. Broad topics are at level 0, and some broad topics split into narrower child topics at level 1. A **terminal topic** has no children. It can be either a child topic or a broad topic that stands alone.

Each topic has a title and description generated from a sample of its chunks. Titles and descriptions help you decide where to explore. They don't define the group, and they might not reflect everything that the group's chunks say. The source chunks are still the evidence.

A **centroid** is a vector that represents the center of a group. Think of it as the topic's typical point among the chunk embeddings. The topic index stores centroids for terminal topics. This notebook derives centroids for broad topics from their children in the same way as the graph explorer. Because a centroid is comparable to chunk embeddings, you can use it to find a chunk's nearest topic or the chunks closest to a topic's center.

A **Sitemap** is a navigable map of the topic hierarchy. It connects broad topics to narrower topics, so a person or an agent can move from a general area toward specific chunks. A sitemap shows you where to look, but only the source records can confirm what happened. Topics also don't cover every chunk. If a branch is missing, that doesn't prove the documents never mention the subject.

<img src="../../img/diagrams/notebook-03-sitemap.svg" alt="Diagram of the Sitemap's relationships. At the top, the Sitemap, a CommunityGraph, points to two level-0 communities through HasRootCommunity. The first is a broad topic that points to narrow topics A and B, both level-1 communities, through HasSubCommunity. The second is a broad topic with no children. Each terminal topic, narrow topic A, narrow topic B, and the broad topic with no children, points to its centroid Vector through HasCentroidVector. A note says that chunks stay in the source index and that a centroid search finds the chunks nearest each topic." width="920">

**🎯 Mission**

This notebook has two parts, and each has a different purpose.

**Look inside the topic graph** (Steps 2–8). These inspection exercises read the stored clustering results to show how chunks were grouped and how navigation works. They are separate from the `retrieve()` calls you'll make in Steps 9 and 10.
- Check how much of the document set the topics cover.
- Reconstruct the explorer's topic tree from the facts in the topic index.
- Compare one chunk's nearest neighbors across all chunks with the neighbors found after routing through its nearest topic centroid.
- Query the sitemap with GQL.

**See how topics affect `retrieve()` results** (Steps 9 and 10) as you investigate a customer complaint. In these steps, you make the same `retrieve()` calls that an app would make.
- Create a topic knowledge base and ask it which areas of the records to explore.
- Ask the standard knowledge base from Part 1 for the PDF evidence, and then check its references in the source records.

Run Part 1 first, because Step 10 reuses its standard knowledge base. Before opening this notebook, you browsed these topics on the graph explorer's **Topics** tab. This notebook rebuilds that view from the stored data, so you don't need to remember the details.

## Step 1: Load your settings and connect to topics and PDF chunks

You'll use two indexes: `cocoarynth-topics` for the topic hierarchy, titles, descriptions, and centroid vectors, and `cocoarynth-chunks` for the PDF text and embeddings. You'll query both directly to see how the topics relate to the chunks they group.

Run this notebook from the repository root or `src/notebooks`.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
API_VERSION = "2026-11-01-preview"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

In [ ]:
import json
import textwrap
from pathlib import Path

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

TOPICS_INDEX = "cocoarynth-topics"
SOURCES_INDEX = "cocoarynth-chunks"
TOPICS_KNOWLEDGE_SOURCE = f"{TOPICS_INDEX}-knowledge-source"

topics_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, TOPICS_INDEX, search_credential, api_version=API_VERSION
)
sources_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, SOURCES_INDEX, search_credential, api_version=API_VERSION
)
for index_name, client in ((TOPICS_INDEX, topics_client), (SOURCES_INDEX, sources_client)):
    print(f"{index_name}: {client.get_document_count()} documents")

## Look inside the topic graph

Steps 2–8 read the stored results of the topic build so you can look more closely at what the graph explorer showed you: how many chunks the topics cover, the facts that form the hierarchy, the centroid vectors, and the sitemap.

### Step 2: Check topic coverage

Before you interpret the tree, check how much of the data it covers. This prepared build assigned some chunks to topics and left others unassigned. A topic's size means more once you know how many chunks were assigned, and at which level.

The [restore manifest](../../data/graph-data/manifest.json) records the coverage counts for the prepared topic build. Run the cell below to print them, then compare them with the **Topics** tab in the explorer.

The topics cover 97 of the lab's 106 chunks, and the other 9 aren't assigned to any topic. Every assigned chunk belongs to one of the four broad topics. Of those, 61 also belong to one of four narrower child topics. Two of the broad topics have no children, so they're terminal topics.

When you select a topic in the explorer, it lists the chunks nearest that topic's centroid from anywhere in the source index. These chunks help you see what the topic is about; they aren't a replay of the build assignments. Later steps use centroid searches to examine how topic routing changes the nearest neighbors.

In [ ]:
metadata_file = Path("data") / "graph-data" / "manifest.json"
if not metadata_file.is_file():
    metadata_file = Path("..") / ".." / metadata_file
build_metadata = json.loads(metadata_file.read_text(encoding="utf-8"))["topics"]
assert build_metadata["schema_version"] == "topicgraph-release-v1"

for key in (
    "schema_version",
    "source_document_count",
    "clustered_source_document_count",
    "unassigned_source_document_count",
    "node_coverage",
    "label_stage",
    "assigned_chunks_by_level",
    "terminal_topic_count",
):
    print(f"{key:<34} {build_metadata[key]}")

### Step 3: Reconstruct the topic tree from its facts

The cell below reconstructs the tree from the explorer's **Topics** tab. It collects each topic's properties, links parents to children, and derives parent centroids in the same way as the explorer. Compare its printed hierarchy with the explorer.

Each `Community` is a topic, and its facts describe the topic's properties and links. Use `predicate` to identify each kind of fact:

* `HasRootCommunity` connects the graph to its level-0 roots; `HasSubCommunity` connects parents to children.
* `prop:title` and `prop:description` use `object_string`; `prop:level`, `prop:proportion`, and `prop:parentProportion` use `object_double`.
* `HasCentroidVector` stores a leaf's `object_vector`.

A topic's count, printed next to its title, is its proportion multiplied by the number of clustered source chunks (97, not all 106), rounded.

Use a topic's `community:<n>` ID to look it up in later queries. A **terminal topic** has no children: it can be a narrower child or one of the two broad topics that stand alone.

In [ ]:
topic_facts = list(topics_client.search(
    search_text="*",
    select=["subject_type", "subject_id", "predicate", "object_id", "object_string", "object_double", "object_vector"],
    top=1000,
))
topics_by_id = {}
property_fields = {
    "prop:title": ("label", "object_string"),
    "prop:description": ("description", "object_string"),
    "prop:level": ("level", "object_double"),
    "prop:proportion": ("proportion", "object_double"),
    "HasCentroidVector": ("centroid", "object_vector"),
}
for fact in topic_facts:
    if fact["subject_type"] == "Community":
        topic = topics_by_id.setdefault(fact["subject_id"], {"topic_id": fact["subject_id"], "parent_id": None, "children": []})
        if fact["predicate"] in property_fields:
            name, field = property_fields[fact["predicate"]]
            topic[name] = fact[field]
for fact in topic_facts:
    if fact["predicate"] == "HasSubCommunity":
        topics_by_id[fact["subject_id"]]["children"].append(fact["object_id"])
        topics_by_id[fact["object_id"]]["parent_id"] = fact["subject_id"]


def leaf_topics(topic):
    return [leaf for child in topic["children"] for leaf in leaf_topics(topics_by_id[child])] if topic["children"] else [topic]


for topic in topics_by_id.values():
    topic["level"] = int(topic["level"])
    topic["member_count"] = round(topic["proportion"] * build_metadata["clustered_source_document_count"])
    topic["is_terminal"] = not topic["children"]
    if not topic["is_terminal"]:
        leaves = leaf_topics(topic)
        total = sum(leaf["proportion"] for leaf in leaves)
        topic["centroid"] = [
            sum(leaf["proportion"] / total * leaf["centroid"][i] for leaf in leaves)
            for i in range(build_metadata["dimension"])
        ]

for parent in sorted((t for t in topics_by_id.values() if t["level"] == 0), key=lambda t: t["topic_id"]):
    print(f"{parent['label']} ({parent['member_count']} chunks; terminal={parent['is_terminal']})")
    print(f"  {parent['description']}")
    for child_id in sorted(parent["children"]):
        child = topics_by_id[child_id]
        print(f"  └─ {child['label']} ({child['member_count']} chunks)")

### Step 4: Pick a chunk to start from

Steps 4–7 follow one chunk: you'll find its nearest neighbors across all chunks, then its neighbors within its nearest topic, and compare the two lists. Find a chunk that mentions Madagascar bean lot MG26-L02 and reuse its stored embedding. This asks "what's near this chunk?" rather than embedding a new question, so no embedding model is called.

In [ ]:
start_chunk = list(sources_client.search(
    search_text='"MG26-L02"',
    search_fields=["chunk"],
    select=["chunk_id", "title", "blob_path", "chunk", "text_vector"],
    top=1,
))[0]
query_vector = start_chunk["text_vector"]
print(f"{start_chunk['title']} ({start_chunk['blob_path']})")
print(textwrap.indent(textwrap.shorten(start_chunk["chunk"], 400), "    "))

### Step 5: Find the nearest chunks directly

Run a vector search over all source chunks using the starting chunk's embedding. The first result is usually the starting chunk itself.

In [ ]:
direct = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

for rank, result in enumerate(direct, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

### Step 6: Find the nearest topic, then search within it

Now find the chunk's nearest topic and look for its neighbors within that topic. The cell below runs three separate vector searches:

1. Find the leaf centroid closest to the starting chunk's embedding, filtering on `predicate eq 'HasCentroidVector'` and searching `object_vector`. Choose among all terminal topics, including the two broad roots at level 0.
2. Query `cocoarynth-chunks` for that centroid's nearest chunks, up to the topic's count (capped at 1,000). These approximate the topic's core.
3. Run the original chunk-vector query again, filtered to those candidates.

This low-level exercise shows how a topic's centroid relates to the chunks it represents. It isn't meant as a way to search for answers: it adds two vector queries and can miss relevant chunks.

In [ ]:
nearest_centroid = list(topics_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="object_vector", k_nearest_neighbors=1, exhaustive=True)],
    filter="predicate eq 'HasCentroidVector'",
    select=["subject_id"],
    top=1,
))[0]
nearest_topic = topics_by_id[nearest_centroid["subject_id"]]
member_count = min(nearest_topic["member_count"], 1000)
member_ids = [
    member["chunk_id"]
    for member in sources_client.search(
        vector_queries=[VectorizedQuery(vector=nearest_topic["centroid"], fields="text_vector", k_nearest_neighbors=member_count, exhaustive=True)],
        select=["chunk_id"],
        top=member_count,
    )
]
routed = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    filter=f"search.in(chunk_id, '{','.join(member_ids)}', ',')",
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

print(f"Nearest topic: {nearest_topic['label']} (level {nearest_topic['level']}; {len(member_ids)} nearest chunks)")
print(f"Starting chunk is among these candidates: {start_chunk['chunk_id'] in member_ids}\n")
for rank, result in enumerate(routed, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

### Step 7: Compare the two lists

The direct search ran one vector query. The topic-routed search ran three: nearest leaf centroid, nearest source chunks for that centroid, and the original query within those candidates.

Compare the overlap to see how limiting the search to centroid-nearest candidates changes the returned neighbors. The candidate set approximates the topic's core; this comparison does not measure which list is more relevant.

In [ ]:
direct_ids = {result["chunk_id"] for result in direct}
routed_ids = {result["chunk_id"] for result in routed}

print(f"Direct search:        {len(direct)} chunks (searched all chunks)")
print(f"Topic-routed search:  {len(routed)} chunks (searched {len(member_ids)} centroid-nearest candidates)")
print(f"\nIn both lists:        {len(direct_ids & routed_ids)}")
print(f"Only in direct list:  {len(direct_ids - routed_ids)}")
print(f"Only in topic list:   {len(routed_ids - direct_ids)}")

### Step 8: Query the sitemap with GQL

The graph profile exposes `CommunityGraph → HasRootCommunity → Community`, `Community → HasSubCommunity → Community`, and `Community → HasCentroidVector → Vector`.

Use GQL to find a broad topic by its `community:<n>` ID, follow it to its children, and read their descriptions. Then run a separate search of the source index with each child's centroid to find example PDFs. Compare the topic names with the content of those documents.

In [ ]:
parent = next(topic for topic in topics_by_id.values() if not topic["is_terminal"])
PARENT_TOPIC_ID = parent["topic_id"]
SITEMAP_GQL = f"""
MATCH (parent:`Community`)-[:`HasSubCommunity`]->(topic:`Community`)
WHERE parent.id = '{PARENT_TOPIC_ID}'
RETURN parent.title AS parent, topic.id AS topic_id, topic.title AS topic, topic.description AS description
LIMIT 50
"""
sitemap_result = topics_client.graph(
    gql=SITEMAP_GQL,
    include_statistics=True,
)
for row in sitemap_result.value:
    print(f"{row['parent']} > {row['topic']}: {row['description']}")
    child = topics_by_id[row["topic_id"]]
    examples = sources_client.search(
        vector_queries=[VectorizedQuery(vector=child["centroid"], fields="text_vector", k_nearest_neighbors=min(child["member_count"], 1000), exhaustive=True)],
        select=["title", "blob_path"], top=2,
    )
    for example in examples:
        print(f"  Centroid-nearest PDF example (separate source search): {example['blob_path']}")

## See how topics affect `retrieve()` results

Now use `KnowledgeBaseRetrievalClient.retrieve()`, the call an app makes to ask a knowledge base a question. Step 9 creates a topic knowledge base and asks it where to look. Step 10 asks the standard knowledge base from Part 1 for the PDF evidence. Compare what each answer draws on.

### Step 9: Find a starting point for a customer complaint

Larkspur Market has complained about grey, dusty-looking Ghana bars. Before deciding what to do, a quality lead needs to find records about the batch, its delivery, and the response to the customer. Ask the topic knowledge base which areas to explore.

`cocoarynth-topics-knowledge-source` connects a knowledge base to the topic index. Its `graphOntology` contains the inline JSON-LD serialization of the [sitemap OWL](../../data/graph-data/cocoarynth-sitemap.owl), which describes the topic types and relationships used by graph search.

![Diagram: cocoarynth-topics-knowledge-base uses cocoarynth-topics-knowledge-source to search CommunityGraph and Community facts with the inline sitemap ontology. Leaf centroids support vector searches over cocoarynth-chunks.](../../img/diagrams/notebook-03.svg)

Create a knowledge base over this source and ask where to look. Its answer instructions tell the model to use only the returned topic titles and descriptions. Read its topic suggestions and activity log. The planner can use graph search or text search to find relevant titles and descriptions. When it uses graph search, the printed `graphSearch` activity includes the generated query because the request sets *graphParameters* to `{"includeQuery": True}`. If no `graphSearch` step appears, the planner used text search this time. Use the suggestions to orient the investigation; in Step 10, you'll check the original records for the business answer.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model, KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort, KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)
topic_source = index_client.get_knowledge_source(TOPICS_KNOWLEDGE_SOURCE).as_dict()
topic_ontology = topic_source["searchIndexParameters"].get("graphOntology", {})
print(f"Inline sitemap ontology: {topic_ontology['name']}")

TOPICS_KB_NAME = "cocoarynth-topics-knowledge-base"
index_client.create_or_update_knowledge_base(KnowledgeBase(
    name=TOPICS_KB_NAME,
    description="Navigate the canonical Cocoarynth topic hierarchy using its inline OWL schema",
    knowledge_sources=[KnowledgeSourceReference(name=TOPICS_KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT, deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
            model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME, api_key=AZURE_OPENAI_KEY,
        )),
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Use returned topic titles and descriptions only and cite reference IDs. Do not invent PDF membership or business facts.",
))

def investigation_request(question, knowledge_source_names, include_graph_query=False):
    source_params = [SearchIndexKnowledgeSourceParams(
        knowledge_source_name=name, include_references=True,
        include_reference_source_data=True, always_query_source=True,
    ) for name in knowledge_source_names]
    if include_graph_query:
        for source_param in source_params:
            if source_param.knowledge_source_name == TOPICS_KNOWLEDGE_SOURCE:
                source_param["graphParameters"] = {"includeQuery": True}
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        include_activity=True,
        knowledge_source_params=source_params,
    )


DISCOVERY_QUESTION = (
    "I'm investigating Larkspur Market's complaint about grey, dusty Ghana bars. "
    "Which areas of our records should I explore to assess the batch, "
    "what happened during delivery, and our response to the customer?"
)
topics_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=TOPICS_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
topic_result = topics_kb_client.retrieve(investigation_request(
    DISCOVERY_QUESTION, [TOPICS_KNOWLEDGE_SOURCE],
    include_graph_query=True,
))
display(Markdown(topic_result.response[0].content[0].text))

print("Activity:")
for activity in topic_result.activity or []:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")
    if step["type"] == "graphSearch":
        print(json.dumps(step, indent=2))

### Step 10: Investigate the complaint in the source records

Now ask the standard knowledge base from Part 1 whether batch GH26-0706 itself was bad or only Larkspur's delivery, whether other stores should pull their bars, and what's already been done for Larkspur. This is a separate request: it sends only the complaint question and searches the production, sales, and policy PDFs. Use the topic suggestions from Step 9 to decide which records to look for in the answer.

Read the answer and its references. Which records describe retained samples, which describe delivery conditions, and which record the customer's credit or replacement? Check the cited evidence before deciding what the findings mean for other stores.

In [ ]:
STANDARD_KB_NAME = "cocoarynth-standard-knowledge-base"
document_knowledge_sources = [
    f"cocoarynth-{domain}-knowledge-source" for domain in ("production", "sales", "policies")
]
COMPLAINT_QUESTION = (
    "Larkspur Market received grey, dusty Ghana bars from batch GH26-0706. "
    "Was the batch itself bad, or just that delivery? "
    "Should other stores that received this batch pull their bars, "
    "and what have we already done for Larkspur?"
)
standard_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=STANDARD_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
document_result = standard_client.retrieve(investigation_request(
    COMPLAINT_QUESTION, document_knowledge_sources,
))
display(Markdown(document_result.response[0].content[0].text))

print("Activity:")
for activity in document_result.activity or []:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in document_result.references or []:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "blob_path" in source_data:
        print(f"    PDF: {source_data['blob_path']}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 400), "      "))

## Choose the right approach for the question

Your two `retrieve()` calls drew on different evidence. The topic knowledge base searched the topic index, so its answer could point you to areas of the records. The standard knowledge base searched the PDF chunks, so its answer could cite the records themselves.

| Need | Good starting point | Tradeoff |
| --- | --- | --- |
| Ground a direct question in original documents | Standard knowledge base | The model plans queries, picks sources, and writes a cited answer |
| Retrieve chunks quickly | Microsoft Retriever 1 (MR1) | Faster planning without a chat completion model, but it searches every source and returns chunks, not an answer |
| Follow explicit relationships across records | Entity graph and GQL | Inspectable joins and provenance, but depends on the quality of the stored relationships |
| Explore groups of related chunks | Topics | Similarity-based browsing, topic hierarchy, and sitemap GQL |

Think about the three questions from this lab: the organic seal, the farms behind batch EC26-0811, and Larkspur's complaint. Which approach would you start with for each, and why?

## ✅ Mission complete

**What you did:**

* ✓ **Topic coverage**: Checked how many source chunks belong to the topic groups.
* ✓ **Topic hierarchy**: Reconstructed the explorer's topic tree from the facts in the topic index.
* ✓ **Topic centroids**: Compared a chunk's nearest neighbors across all chunks with those in its nearest topic.
* ✓ **Sitemap**: Queried a broad topic's children with GQL and found PDF examples through separate centroid searches of the source index.
* ✓ **Customer investigation**: Used the topic knowledge base to find areas to explore, then asked the standard knowledge base for the PDF evidence behind Larkspur's complaint.

➡️ Return to lab instructions.